# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring the **Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells** dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id`s.

In [ ]:
# List all available record sets, their fields, and columns
record_sets = dataset.record_sets
print(f"Available Record Sets: {len(record_sets)}\n")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}\n  @id: {rs.id}\n  Description: {rs.description}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'N/A')})")
    if hasattr(rs, 'columns'):
        print("  Columns:")
        for column in rs.columns:
            print(f"    - {column.name} (@id: {column.id}, type: {getattr(column, 'data_type', 'N/A')})")
    print("")

## 3. Data Extraction
Load data from all record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in dataset.record_sets]

# Extract all data to DataFrames
dataframes = {}
for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for record_set @id: {rs_id}, shape: {df.shape}")

# Example: Show columns and first few records of the main table (use first record set as example)
main_record_set_id = record_set_ids[0]
print("\nColumns in main DataFrame:")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data. This section leverages field IDs from the actual Croissant schema.

In [ ]:
# Inspect available fields for EDA
df = dataframes[main_record_set_id]
print("First few column names:", df.columns[:10].tolist())

# Let's try to find a numeric field (e.g. molecular weight, coverage etc.)
# We'll try a common field name, otherwise change to a correct column name available in the columns
possible_numeric_fields = [
    'MW', 'Molecular_weight', 'Coverage', 'Normalized abundance', 
    'PeptideCount', 'pI', 'Abundance', 'coverage', 'Peptides', 'SpectralCount'
]
numeric_field = None
for col in df.columns:
    if any(key.lower() in col.lower() for key in possible_numeric_fields):
        numeric_field = col
        break
if numeric_field is None:
    # fallback to any numeric dtype
    numeric_candidates = df.select_dtypes(include=['float', 'int']).columns.tolist()
    if numeric_candidates:
        numeric_field = numeric_candidates[0]
    else:
        print("No numeric fields found for EDA.")
        numeric_field = df.columns[0]

print(f"Using numeric field: {numeric_field}")
# Drop records with missing values in the main numeric field
filtered_df = df[pd.to_numeric(df[numeric_field], errors='coerce').notnull()].copy()
filtered_df[numeric_field] = pd.to_numeric(filtered_df[numeric_field], errors='coerce')

# Simple numeric filter: e.g., select where value > 10 (adjust as appropriate for the field)
threshold = 10
filtered_df = filtered_df[filtered_df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold} (n={len(filtered_df)}):")
print(filtered_df.head())

# Normalize
filtered_df[f"{numeric_field}_normalized"] = (
    filtered_df[numeric_field] - filtered_df[numeric_field].mean()
) / (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() else 1)
print(f"\nNormalized {numeric_field}:\n", filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a categorical field, e.g., 'description' or 'Gene' or similar
possible_group_fields = [
    'Description', 'Gene', 'Protein', 'Sample', 'Group', 'Accession', 'accession', 
    'ProteinName', 'GroupID', 'SampleID', 'Protein Description'
]
group_field = None
for col in df.columns:
    if any(key.lower() in col.lower() for key in possible_group_fields):
        group_field = col
        break

if group_field is not None
    and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df.head())

## 5. Visualization
Visualize the distribution of the main numeric field and show group means if applicable.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field], kde=True, bins=30)
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field} (>{threshold})")
plt.show()

# If grouped by a category, show mean per group (top 10)
if 'grouped_df' in locals():
    plt.figure(figsize=(12, 4))
    top_groups = grouped_df.sort_values(numeric_field, ascending=False).head(10)
    sns.barplot(x=group_field, y=numeric_field, data=top_groups)
    plt.title(f"Top 10 {group_field} by mean {numeric_field}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

## 6. Conclusion

- We have explored the FAIR^2 Croissant dataset using the `mlcroissant` package and listed available record sets and fields by their `@id`.
- Main numeric fields (e.g., molecular weight, coverage, or abundance) were filtered, normalized, and visualized.
- Grouped summaries were computed for a selected categorical attribute, if available.

This workflow can be adapted for further processing, analysis, or integration into ML pipelines on mass spectrometry protein datasets and other Croissant-compatible resources.